# 05 LambdaMART: ablations, collaborative features, final models

**Purpose.** Train gradient-boosted rankers (LightGBM LambdaMART) one step at a time and measure every step against the same references on the same searches: a deployable non-personalised base, which history representation works, which further feature groups help, item-item collaborative features (CF), the final models on all training searches, a robustness check and sensitivity checks. The last section scores the test searches for notebook 07 (without looking at any test label).

**Reference.** Lagged history popularity (`pop_history_lag1d`, history actions at least one day old): about 0.337 NDCG@10 on `validation_clean` (notebook 04). A gain counts as meaningful only if it is **at least +0.003 with a paired 95% bootstrap interval excluding 0**; smaller effects are reported but not acted on.

**Rules.**
- Inputs only from the groups in `features_v3_meta.json`; never `session_id`, `wid`, `label` or the list length.
- Training searches: without a clamped final gap (post-search actions in the history), without an engaged candidate (every order scores the same) and without single-candidate lists.
- Early stopping on an **inner set** of 10% of the dev training searches (whole searches, seed 4222), so validation is used only to report, never to pick the number of trees.
- LightGBM LambdaRank with **linear gains** `label_gain=[0,1,2,3]` (LightGBM's default gains are exponential) and NDCG@10 for early stopping; reported numbers come from our exact tie-aware evaluator on complete validation lists.
- Main population `validation_clean`; all validation searches as sensitivity. Comparisons are paired per-search differences with bootstrap intervals over searches.
- Candidates were shuffled within each search when the features were built (notebook 03), so file order, which encodes the label, cannot reach a model.
- Missing values (unknown metadata) stay missing: LightGBM learns where to send them, instead of treating unknown as "no match".
- **Screening vs final:** sections 3-10 train on the 40,000-search dev subsample (screening); section 11 retrains the finalists on all training searches, and those are the reported numbers.

**Runtime:** high-RAM CPU runtime, about 3-4 hours without the model cache. Every trained model is cached in the working folder (`processed/models/`), so a re-run after a disconnect continues where it stopped.

In [ ]:
# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup(packages={"lightgbm": "4.6.0"})  # pinned: tree models must be reproducible
jd.require(paths, "data/user_behavior_data.txt", "processed/search_table.parquet", "processed/split_v1.csv",
           *[f"processed/features_v3_{split}.parquet" for split in ("train_dev", "train_rest", "validation", "test")],
           "processed/features_v3_meta.json", "processed/features_v3_minage0.00694444d_meta.json",
           *[f"processed/features_v3_minage0.00694444d_{split}.parquet" for split in ("train_dev", "validation")])

In [ ]:
import gc
import hashlib
import json
import os
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from scipy import sparse

from jdsearch import SECONDS_PER_DAY, evaluate, iter_records, paired_difference, summarise

IN_COLAB = jd.IN_COLAB
PROCESSED = paths.processed
BEHAVIOUR = paths.behaviour  # raw records, only for the collaborative features (section 10)
pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v3"
SEED = 4222        # data decisions (inner split)
MODEL_SEED = 1     # LightGBM
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]
ALL_FEATURES = [f for group in meta["groups"].values() for f in group]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES)) == 97
assert not set(ALL_FEATURES) & {"session_id", "wid", "label", "n_candidates"}
print(f"LightGBM {lgb.__version__}; {len(ALL_FEATURES)} features in {len(meta['groups'])} groups")

## 1. Training population and inner early-stopping set

In [ ]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
drops = pd.DataFrame({
    "clamped final gap": dev_info.final_gap_clamped,
    "no engaged candidate": dev_info.n_engaged == 0,
    "single candidate": dev_info.n_candidates == 1,
})
train_pool = dev_ids[~drops.any(axis=1).to_numpy()]
rng = np.random.default_rng(SEED)
inner_ids = np.sort(rng.choice(train_pool.to_numpy(), size=round(0.1 * len(train_pool)), replace=False))
fit_ids = np.setdiff1d(train_pool.to_numpy(), inner_ids)
print(f"dev searches {len(dev_ids):,}; kept for training {len(train_pool):,} "
      f"(fit {len(fit_ids):,}, inner early-stopping set {len(inner_ids):,})")
drops.sum().rename("dropped searches (overlapping)").to_frame()

## 2. Loading and memory profile

Features are read **column by column into one pre-allocated float32 matrix** (column-major, which LightGBM accepts without copying), so no intermediate DataFrame of the full table is built. The rows of each search are contiguous in the feature files; LightGBM's `group` is the list of consecutive search sizes. Matrices above 1.5 GB (all training searches) are memory-mapped.

The profile loads all 97 features for the dev subsample and validation and builds a LightGBM dataset from them. "Peak" is the highest memory the process has used so far.

**Model cache.** Every trained model is saved under `processed/models/<key>` (booster at its best iteration, validation scores, summary). The key covers the feature list, LightGBM settings, training and inner searches, feature-file sizes and dates and the LightGBM version, so any change trains a new model; re-running the notebook only trains what is new. `RETRAIN = True` ignores the cache.

In [ ]:
PROCESS = psutil.Process()


def memory_gb():
    """(current, peak) resident memory of this process in GB."""
    info = PROCESS.memory_info()
    peak = getattr(info, "peak_wset", None)  # Windows
    if peak is None:  # Linux / Colab: ru_maxrss is in KiB
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    return info.rss / 1e9, peak / 1e9


LEVELS = ["product", "shop_other", "brand_other", "L3"]
# Derived inputs: sums of v3 columns, built while loading (section 8: all action types of the last 7 days pooled).
DERIVED = {f"recent_0_7_{level}": [f"{block}_{level}" for block in ("click_0_7", "cart_0_7", "ord_0_7")]
           for level in LEVELS}
# Inputs stored outside the v3 files (section 10: collaborative features): column -> function(split, version) -> file
# with the same rows in the same order as features_{version}_{split}.parquet.
EXTRA = {}


MEMMAP_GB = 1.5  # larger input matrices go to a memory-mapped file on disk (all training searches)
MEMMAP_DIR = Path("/content/tmp") if IN_COLAB else PROCESSED / "tmp"
memmap_files = []


def feature_file(split, version=None):
    """Feature file of a split; `version` defaults to FEATURES (the other version is the safety-gap build, section 12)."""
    return PROCESSED / f"features_{version or FEATURES}_{split}.parquet"


def load_features(splits, columns, sessions=None, version=None, with_labels=True):
    """Session ids, float32 feature matrix (column-major) and labels; rows restricted to `sessions` if given.

    `splits` is one split name or several, whose rows are stacked in that order (every search stays contiguous). A
    column in DERIVED is the sum of its v3 parts (missing if a part is missing); a column in EXTRA is read from its own
    file. Matrices above MEMMAP_GB are memory-mapped, so the operating system can page them out. With
    `with_labels=False` the label column is not read (test scoring) and None is returned for the labels.
    """
    splits = [splits] if isinstance(splits, str) else list(splits)
    selected = []
    for split in splits:
        ids = pq.read_table(feature_file(split, version), columns=["session_id"]).column(0).to_numpy()
        if not (np.diff(ids) >= 0).all():
            raise ValueError("Rows must be sorted by session_id so that every search is contiguous")
        selected.append((split, ids, np.ones(len(ids), dtype=bool) if sessions is None else np.isin(ids, sessions)))
    shape = (sum(int(keep.sum()) for _, _, keep in selected), len(columns))
    if shape[0] * shape[1] * 4 / 1e9 > MEMMAP_GB:
        MEMMAP_DIR.mkdir(parents=True, exist_ok=True)
        memmap_files.append(MEMMAP_DIR / f"inputs_{os.getpid()}_{len(memmap_files)}.npy")
        matrix = np.lib.format.open_memmap(memmap_files[-1], mode="w+", dtype=np.float32, shape=shape, fortran_order=True)
    else:
        matrix = np.empty(shape, dtype=np.float32, order="F")
    offset, labels = 0, []
    for split, ids, keep in selected:
        n = int(keep.sum())
        for j, column in enumerate(columns):
            source = EXTRA[column](split, version) if column in EXTRA else feature_file(split, version)
            parts = DERIVED.get(column, [column])
            matrix[offset:offset + n, j] = sum(pq.read_table(source, columns=[p]).column(0).to_numpy() for p in parts)[keep]
        if with_labels:
            labels.append(pq.read_table(feature_file(split, version), columns=["label"]).column(0).to_numpy()[keep])
        offset += n
    return (np.concatenate([ids[keep] for _, ids, keep in selected]), matrix,
            np.concatenate(labels) if with_labels else None)


def remove_memmaps():
    """Delete memory-mapped input files that are no longer in use."""
    for f in list(memmap_files):
        try:
            f.unlink()
            memmap_files.remove(f)
        except (PermissionError, FileNotFoundError):
            pass


def group_sizes(ids):
    """Sizes of consecutive runs of equal session ids (LightGBM's `group`)."""
    bounds = np.r_[0, np.flatnonzero(np.diff(ids)) + 1, len(ids)]
    return np.diff(bounds)

In [ ]:
profile = [("start", *memory_gb())]
start = time.time()
train_sid, train_X, train_y = load_features("train_dev", ALL_FEATURES, sessions=train_pool)
profile.append(("train features loaded (97 columns)", *memory_gb()))
categorical = [ALL_FEATURES.index(c) for c in meta["categorical"]]
probe = lgb.Dataset(train_X, label=train_y, group=group_sizes(train_sid), categorical_feature=categorical,
                    free_raw_data=False, params={"verbose": -1, "feature_pre_filter": False}).construct()
profile.append(("LightGBM dataset built", *memory_gb()))
valid_sid, valid_X, valid_y = load_features("validation", ALL_FEATURES)
profile.append(("validation features loaded", *memory_gb()))
seconds = time.time() - start
train_rows, train_gb = train_X.shape[0], train_X.nbytes / 1e9
del probe, train_X, valid_X, train_sid, valid_sid, train_y, valid_y
gc.collect()
profile.append(("all freed", *memory_gb()))
profile = pd.DataFrame(profile, columns=["step", "current_gb", "peak_gb"]).set_index("step")
full_rows = train_rows * (assignment.split == "train").sum() / len(dev_ids)
print(f"training rows {train_rows:,} ({train_gb:.2f} GB as float32); load + build {seconds:.0f} s")
print(f"estimate for all training searches: {full_rows / 1e6:.1f}M rows, "
      f"{full_rows * len(ALL_FEATURES) * 4 / 1e9:.1f} GB as float32 before LightGBM's dataset")
print(f"machine memory: {psutil.virtual_memory().total / 1e9:.1f} GB total")
profile

## 3. Deployable non-personalised base (G0 + G1 + G2a)

Ten features: query length (G0, acts only through interactions because it is constant within a list), the eight query-title relevance features (G1) and lagged history popularity (G2a). Nothing in the base uses the shopper's history or same-day information, so it is the trained version of the deployable reference.

One fixed LightGBM configuration is used for all models below; it is not tuned. Success criterion: the lower bound of the paired 95% interval against lagged popularity on `validation_clean` is above +0.003.

In [ ]:
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [10],
    "label_gain": [0, 1, 2, 3],          # linear gains, as in our evaluator
    "lambdarank_truncation_level": 20,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_data_in_leaf": 500,
    "feature_pre_filter": False,
    "seed": MODEL_SEED,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
MAX_ROUNDS, PATIENCE = 3000, 100
BASE = meta["groups"]["G0_search"] + meta["groups"]["G1_relevance"] + meta["groups"]["G2a_popularity_deployable"]


RETRAIN = False  # True: ignore cached models and train everything again
# Bump CACHE_VERSION after changing training or feature-loading code, or meta["categorical"] (not in the key otherwise);
# bump CF_SPEC["version"] after changing the CF code (section 10). Version 1 leaves the existing keys unchanged.
CACHE_VERSION = 1
CACHE = PROCESSED / "models"
FEATURE_FILES = [feature_file(split) for split in ("train_dev", "validation")]
validation_frame = pq.read_table(FEATURE_FILES[1], columns=["session_id", "label"]).to_pandas()


def digest(ids):
    return hashlib.sha1(np.sort(np.asarray(ids, dtype=np.int64)).tobytes()).hexdigest()


def file_stats(f):
    return [f.name, f.stat().st_size, int(f.stat().st_mtime)]


def cache_key(columns, params, fit_sessions, train_splits=("train_dev",), version=None):
    version = version or FEATURES
    spec = {"features": version, "columns": columns, "derived": {c: DERIVED[c] for c in columns if c in DERIVED},
            "params": params, "rounds": [MAX_ROUNDS, PATIENCE], "fit": digest(fit_sessions), "inner": digest(inner_ids),
            "files": [file_stats(feature_file(s, version)) for s in ("train_dev", "validation")],
            "lightgbm": lgb.__version__}
    extra_splits = sorted({*train_splits, "train_dev", "validation"})
    extra = {c: [file_stats(EXTRA[c](s, version)) for s in extra_splits] for c in columns if c in EXTRA}
    if extra:  # only models with extra inputs (CF) get this entry, so earlier keys stay valid
        spec["extra"] = extra
    if CACHE_VERSION != 1:
        spec["cache_version"] = CACHE_VERSION
    if tuple(train_splits) != ("train_dev",):  # likewise only models trained on more than the dev subsample
        spec["train_splits"] = [file_stats(feature_file(s, version)) for s in train_splits]
    return hashlib.sha1(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]


def train_ranker(columns, params=PARAMS, label="", fit_sessions=None, train_splits=("train_dev",), version=None):
    """Fit on the fit searches (or a subset, or searches from several training splits), stop early on the inner set,
    score every validation candidate.

    Returns the booster (best iteration), a summary and the validation scores (session_id, label, score), from the
    cache if this exact model was trained before.
    """
    fit_sessions = fit_ids if fit_sessions is None else fit_sessions
    folder = CACHE / cache_key(columns, params, fit_sessions, train_splits, version)
    if not RETRAIN and (folder / "summary.json").is_file():
        summary = json.loads((folder / "summary.json").read_text())
        return (lgb.Booster(model_file=str(folder / "model.txt")), {**summary, "cached": True},
                validation_frame.assign(score=np.load(folder / "validation_scores.npy")))
    cats = [columns.index(c) for c in meta["categorical"] if c in columns]
    sid, X, y = load_features(train_splits, columns, sessions=fit_sessions, version=version)  # no copies of X
    fit_set = lgb.Dataset(X, label=y, group=group_sizes(sid), feature_name=columns, categorical_feature=cats)
    sid_inner, X_inner, y_inner = load_features("train_dev", columns, sessions=inner_ids, version=version)
    inner_set = lgb.Dataset(X_inner, label=y_inner, group=group_sizes(sid_inner), reference=fit_set)
    start = time.time()
    booster = lgb.train(params, fit_set, num_boost_round=MAX_ROUNDS, valid_sets=[inner_set], valid_names=["inner"],
                        callbacks=[lgb.early_stopping(PATIENCE, verbose=False), lgb.log_evaluation(200)])
    summary = {"model": label, "features": len(columns), "fit_searches": len(fit_sessions),
               "best_iteration": int(booster.best_iteration),
               "inner_ndcg_at_10": float(booster.best_score["inner"]["ndcg@10"]),
               "train_seconds": round(time.time() - start), "peak_memory_gb": round(memory_gb()[1], 2)}
    del X, X_inner, fit_set, inner_set
    gc.collect()
    remove_memmaps()
    sid, X, _ = load_features("validation", columns, version=version)
    assert (sid == validation_frame.session_id.to_numpy()).all()
    scores = booster.predict(X, num_iteration=booster.best_iteration)
    del X
    folder.mkdir(parents=True, exist_ok=True)
    booster.save_model(str(folder / "model.txt"), num_iteration=booster.best_iteration)
    np.save(folder / "validation_scores.npy", scores)
    (folder / "columns.json").write_text(json.dumps(columns))
    (folder / "summary.json").write_text(json.dumps(summary))
    return booster, {**summary, "cached": False}, validation_frame.assign(score=scores)


base_model, base_summary, base_scores = train_ranker(BASE, label="LambdaMART base G0+G1+G2a")
pd.Series(base_summary).to_frame("base")

In [ ]:
scored = base_scores.copy()
scored["pop_lag1d"] = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                                    columns=["pop_history_lag1d"]).column(0).to_numpy()
per_search = {
    "LambdaMART base": evaluate(scored, "score", expected_sizes=validation_sizes),
    "lagged history popularity": evaluate(scored.assign(ref=scored.pop_lag1d.fillna(0).astype(float)), "ref",
                                          expected_sizes=validation_sizes),
    "random order": evaluate(scored.assign(constant=0.0), "constant", expected_sizes=validation_sizes),
}
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]

# Sanity check: the reference reproduces its value from notebook 04 (validation_clean).
assert abs(per_search["lagged history popularity"].ndcg.loc[clean_ids].mean()
           - jd.logged(paths, "history popularity, lag 1 day")) < 1e-4

rows = {}
for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
    for name, per in per_search.items():
        row = summarise(per.loc[ids]).iloc[0].to_dict()
        if name != "lagged history popularity":
            d = paired_difference(per.loc[ids], per_search["lagged history popularity"].loc[ids])
            row.update(vs_lagged_pop=d[0], vs_lagged_pop_low=d[1], vs_lagged_pop_high=d[2])
        rows[(population, name)] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

### By segment (validation_clean)

Popularity is much weaker in long, deep-scroll lists, so list length is the segment to watch; activity in the last 7 days separates shoppers with fresh history from returning ones. Each cell is the paired difference base minus lagged popularity within the segment.

In [ ]:
info = searches.loc[clean_ids]
segments = {
    "list length": pd.cut(info.n_candidates, [0, 60, 84, 95, 200], labels=["1-60", "61-84", "85-95", "96-200"]),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=info.index),
}
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search["LambdaMART base"].loc[ids], per_search["lagged history popularity"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

Split importance (share of total gain). This shows what the trees use, not how much shoppers value a signal; correlated features share gain arbitrarily.

In [ ]:
importance = pd.Series(base_model.feature_importance("gain"), index=BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

### Results table

In [ ]:
NOTES = (f"LightGBM lambdarank, features {FEATURES} G0+G1+G2a, linear gains, trunc 20, 31 leaves, lr 0.05, "
         f"min leaf 500, inner early stopping (best iteration {base_summary['best_iteration']}), seed {MODEL_SEED}")
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    jd.log_result(paths, "05", "base", "LambdaMART base G0+G1+G2a", split, per_search["LambdaMART base"].loc[ids],
                  notes=NOTES)
jd.read_results(paths).query("notebook == '05'")

**Result.**
- The trained base reaches about **0.347** NDCG@10 on `validation_clean`, about **+0.010** over lagged history popularity, with the interval above +0.003: grouping, gains and evaluator behave as expected, and the base is the reference for personalisation.
- The gain comes from lists of up to 84 candidates; in lists of more than 95 candidates the base is not distinguishable from popularity. Activity in the last 7 days makes little difference, as expected for a model without personal history.
- Lagged popularity carries most of the split gain, BM25 most of the rest: relevance mainly reorders products of similar popularity.
- Memory: the 97 features of the dev training searches fit comfortably; all training searches (about 11.3M rows) are memory-mapped in section 11.

## 4. Full base with same-day popularity (G0 + G1 + G2a + G2b)

G2b adds the same-day popularity signals: training exposures, the out-of-fold engagement rate and history popularity without the one-day lag. All lists are from one day, so other shoppers' same-day behaviour may come after the search being ranked: the full base is an **optimistic, non-deployable reference**, never a candidate model. It shows how much a model would gain from information it could not have at search time.

In [ ]:
def compare(name, against=("LambdaMART base", "lagged history popularity")):
    """Summary of `name` on both populations with paired differences against the references."""
    rows = {}
    for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
        row = summarise(per_search[name].loc[ids]).iloc[0].to_dict()
        for other in against:
            d = paired_difference(per_search[name].loc[ids], per_search[other].loc[ids])
            row.update({f"vs {other}": d[0], f"vs {other} low": d[1], f"vs {other} high": d[2]})
        rows[population] = row
    table = pd.DataFrame(rows).T
    table["searches"] = table.searches.astype(int)
    return table


def log_model(step, model, split, per, summary, version=None, seed=MODEL_SEED):
    """Write one trained model's result to the results table."""
    jd.log_result(paths, "05", step, model, split, per,
                  notes=(f"LightGBM lambdarank, features {version or FEATURES}, {summary['features']} inputs, "
                         f"{summary['fit_searches']:,} fit searches, inner early stopping "
                         f"(best iteration {summary['best_iteration']}), configuration as the base, seed {seed}"))


FULL_BASE = BASE + meta["groups"]["G2b_popularity_same_day"]
full_base_model, full_base_summary, full_base_scores = train_ranker(FULL_BASE,
                                                                   label="LambdaMART full base G0+G1+G2a+G2b")
per_search["LambdaMART full base"] = evaluate(full_base_scores, "score", expected_sizes=validation_sizes)
display(pd.Series(full_base_summary).to_frame("full base"))
compare("LambdaMART full base")

In [ ]:
importance = pd.Series(full_base_model.feature_importance("gain"), index=FULL_BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

In [ ]:
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    log_model("full base (same-day)", "LambdaMART full base G0+G1+G2a+G2b (same-day, not deployable)", split,
             per_search["LambdaMART full base"].loc[ids], full_base_summary)
jd.read_results(paths).query("step == 'full base (same-day)'")

**Result.**
- The full base reaches about **0.366** on `validation_clean`, about **+0.019** over the deployable base. As single signals, lagged and unlagged history popularity differ by only 0.004 (notebook 04); combined in a model, the same-day signals are worth much more.
- Unlagged history popularity takes most of the split gain and the lagged version almost none: where both are available the trees prefer the version that includes same-day events.
- The full base stays a non-deployable reference: it shows that information from the day of the search is valuable, which is exactly what a deployed ranker cannot use.

## 5. Learning curve of the deployable full model

The **deployable full model** uses every v3 group except G2b (94 features): the base plus history (H1-H5), context (C1, C2), query-history (Q1) and source (S1) features. It is trained on nested random subsets of the dev fit searches (10,000, 20,000 and all; seed 4222), always with the same inner early-stopping set, and scored on the same validation searches.

Two questions: (1) does the model still gain from more data at the size of the dev subsample (if so, the finalists are retrained on all training searches, section 11); (2) as a first look, how much do personal history and context add over the deployable base? The ablations below take this gain apart group by group.

In [ ]:
DEPLOYABLE_FULL = [f for f in ALL_FEATURES if f not in meta["groups"]["G2b_popularity_same_day"]]
curve_order = np.random.default_rng(SEED).permutation(fit_ids)
CURVE_SIZES = [10_000, 20_000, len(fit_ids)]
curve_summaries = {}
for n in CURVE_SIZES:
    booster, summary, scores = train_ranker(DEPLOYABLE_FULL, label=f"deployable full, {n:,} fit searches",
                                            fit_sessions=np.sort(curve_order[:n]))
    name = f"deployable full, {n:,} fit searches"
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    curve_summaries[n] = summary
    if n == len(fit_ids):
        full_model = booster
FULL_NAME = f"deployable full, {len(fit_ids):,} fit searches"

rows = []
for i, n in enumerate(CURVE_SIZES):
    name = f"deployable full, {n:,} fit searches"
    per = per_search[name].loc[clean_ids]
    row = {"fit_searches": n, "best_iteration": curve_summaries[n]["best_iteration"],
           "train_seconds": curve_summaries[n]["train_seconds"], "ndcg_at_10": per.ndcg.mean()}
    d = paired_difference(per, per_search["LambdaMART base"].loc[clean_ids])
    row.update(vs_base=d[0], vs_base_low=d[1], vs_base_high=d[2])
    if i:
        previous = per_search[f"deployable full, {CURVE_SIZES[i - 1]:,} fit searches"].loc[clean_ids]
        d = paired_difference(per, previous)
        row.update(vs_previous=d[0], vs_previous_low=d[1], vs_previous_high=d[2])
    rows.append(row)
curve = pd.DataFrame(rows).set_index("fit_searches")
print(f"peak memory so far {memory_gb()[1]:.2f} GB")
curve

In [ ]:
compare(FULL_NAME)

### Where the deployable full model gains (validation_clean)

Paired difference full model minus deployable base within each segment. Personal history should matter mostly for shoppers active in the last 7 days.

In [ ]:
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search[FULL_NAME].loc[ids], per_search["LambdaMART base"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "full": per_search[FULL_NAME].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

Split gain by feature group (share of the total). A description of what the trees use, not a ranking of signals by value: correlated groups share gain arbitrarily, and the ablations, not importances, decide which groups matter.

In [ ]:
group_of = {f: g for g, features in meta["groups"].items() for f in features}
importance = pd.Series(full_model.feature_importance("gain"), index=DEPLOYABLE_FULL)
by_group = importance.groupby(importance.index.map(group_of)).sum()
display((by_group / by_group.sum()).sort_values(ascending=False).rename("gain share").to_frame())
(importance / importance.sum()).sort_values(ascending=False).head(15).rename("gain share, top 15 features").to_frame()

In [ ]:
for n in CURVE_SIZES:
    name = f"deployable full, {n:,} fit searches"
    log_model("learning curve", f"LambdaMART deployable full (no G2b), {n:,} fit searches", "validation_clean",
             per_search[name].loc[clean_ids], curve_summaries[n])
log_model("learning curve", f"LambdaMART deployable full (no G2b), {len(fit_ids):,} fit searches", "validation",
         per_search[FULL_NAME].loc[all_ids], curve_summaries[len(fit_ids)])
jd.read_results(paths).query("step == 'learning curve'")

**Result.**
- **Personal history and context add about +0.042** to the deployable base (about 0.389 on `validation_clean`), about twice as much for shoppers active in the last 7 days, and unlike the base also in long lists.
- **Learning curve:** the step from 20,000 to all dev fit searches still gains a few thousandths, so the finalists are retrained on all training searches (section 11). One training sample per size cannot separate diminishing returns from noise.
- Lagged popularity still takes most of the split gain, pooled exact-product history most of the rest; importances do not say which groups matter, the ablations do.

## 6. Untrained rules: own products first, then popularity

How much of the deployable full model's gain does a simple rule already reach? The rule puts candidates the shopper interacted with before at the top, then sorts by lagged popularity: the obvious way to use history, with no learning.

- **(a) count:** more past actions on the exact product rank higher (`pooled_product`, all types and ages), lagged popularity breaks ties.
- **(b) any match:** any past action on the exact product counts the same, lagged popularity breaks ties.

Popularity is scaled to below 1 and added to the history count, so the order is strictly lexicographic. A fairer rule that orders the rest by the trained base's score follows in section 15.

In [ ]:
rule_inputs = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                            columns=["session_id", "pooled_product"]).to_pandas()
assert (rule_inputs.session_id.to_numpy() == scored.session_id.to_numpy()).all()  # same row order as `scored`
own = rule_inputs.pooled_product.fillna(0).to_numpy(dtype=float)
pop = scored.pop_lag1d.fillna(0).to_numpy(dtype=float)
pop_part = pop / (pop.max() + 1)  # strictly below 1: popularity only orders candidates with equal history
RULES = {"rule (a): own-product count, then popularity": own + pop_part,
         "rule (b): any own product, then popularity": (own > 0) + pop_part}
for name, score in RULES.items():
    per_search[name] = evaluate(scored.assign(rule=score), "rule", expected_sizes=validation_sizes)

clamped_ids = all_ids.difference(clean_ids)
shown = ["lagged history popularity", "LambdaMART base", *RULES, FULL_NAME]
table = pd.DataFrame({population: {name: per_search[name].ndcg.loc[ids].mean() for name in shown}
                      for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids),
                                              ("clamped only", clamped_ids)]})
pairs = [("rule (a): own-product count, then popularity", "lagged history popularity"),
         ("rule (a): own-product count, then popularity", "LambdaMART base"),
         ("rule (b): any own product, then popularity", "rule (a): own-product count, then popularity"),
         (FULL_NAME, "rule (a): own-product count, then popularity"),
         (FULL_NAME, "rule (b): any own product, then popularity")]
differences = pd.DataFrame(
    {f"{a}  minus  {b}": dict(zip(["difference", "low", "high"],
                                  paired_difference(per_search[a].loc[clean_ids], per_search[b].loc[clean_ids])))
     for a, b in pairs}).T
display(table.rename_axis("NDCG@10"))
differences.rename_axis("validation_clean, paired")

### Where the rule and the trained model differ (validation_clean)

The rule can only differ from popularity in searches whose list contains a product from the shopper's history; everywhere else it **is** lagged popularity. Splitting by that, and by activity in the last 7 days, shows where the trained model gains beyond the rule.

In [ ]:
has_own = pd.Series(own > 0).groupby(scored.session_id.to_numpy()).any()
info = searches.loc[clean_ids]
rule_segments = {
    "list contains an own product": pd.Series(np.where(has_own.reindex(clean_ids), "yes", "no"), index=clean_ids),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=clean_ids),
}
RULE_A = "rule (a): own-product count, then popularity"
rows = []
for segment, labels in rule_segments.items():
    for value in sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search[FULL_NAME].loc[ids], per_search[RULE_A].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "rule_a": per_search[RULE_A].ndcg.loc[ids].mean(),
                     "full_model": per_search[FULL_NAME].ndcg.loc[ids].mean(),
                     "full_minus_rule": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

In [ ]:
entries = [(name, "validation_clean", clean_ids) for name in RULES] + [(RULE_A, "validation", all_ids)]
for name, split, ids in entries:
    jd.log_result(paths, "05", "rules", name, split, per_search[name].loc[ids],
                  notes=f"untrained: sort by pooled_product ({'count' if '(a)' in name else 'any'}), "
                        f"ties by pop_history_lag1d (features {FEATURES})")
jd.read_results(paths).query("step == 'rules'")

**Result.**
- Rule (a) reaches about **0.373** on `validation_clean`, well above lagged popularity and the trained base. Counting past actions helps a little over "any match".
- The deployable full model adds about **+0.016** beyond the rule, both where no candidate is an own product (relevance plus shop, brand and category evidence and context) and where one is (better weighting of own products against popularity).
- On the clamped searches the rule is as good as the trained model: their own-product matches are post-search actions, which confirms excluding them.

## 7. History representation: pooled vs by type vs by type and window

The three history groups hold the **same evidence at three resolutions** (H3 sums to H2, H2 to H1), with the same four match levels (exact product; shop, brand via other products; L3 category path):

| Model | Inputs | Reads as |
|---|---|---|
| M1 | base + H1 pooled (4) | one pool of history |
| M2 | base + H2 by action type (16) | M2 - M1: does the action type matter? |
| M3 | base + H3 by type and window (28) | M3 - M2: does age within a type matter? |
| M3 + H1 | M3 plus the pooled totals | trees cannot add features up; separates "age does not matter" from "split counts are harder to use" |

Decision rule: a finer representation counts only if it is at least +0.003 better with a paired interval excluding 0; otherwise the simpler one is preferred.

In [ ]:
H = meta["groups"]
model_summaries = {}


def run_models(models):
    """Train (or load) each model and evaluate it on all validation lists; return a summary table."""
    for name, columns in models.items():
        _, summary, scores = train_ranker(columns, label=name)
        per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
        model_summaries[name] = summary
    keys = ["features", "best_iteration", "train_seconds", "cached"]
    table = pd.DataFrame({n: {k: model_summaries[n][k] for k in keys} for n in models}).T
    table["ndcg_clean"] = [per_search[n].ndcg.loc[clean_ids].mean() for n in models]
    return table


def paired_table(pairs, ids=None):
    """Paired per-search differences (A minus B) with bootstrap intervals; validation_clean by default."""
    ids = clean_ids if ids is None else ids
    rows = {}
    for a, b in pairs:
        d = paired_difference(per_search[a].loc[ids], per_search[b].loc[ids])
        rows[f"{a}  minus  {b}"] = {"A": per_search[a].ndcg.loc[ids].mean(), "B": per_search[b].ndcg.loc[ids].mean(),
                                    "difference": d[0], "low": d[1], "high": d[2]}
    return pd.DataFrame(rows).T


M1, M2, M3, M3T = "M1 base + H1 pooled", "M2 base + H2 by type", "M3 base + H3 by type and window", "M3 + H1 totals"
HISTORY_MODELS = {M1: BASE + H["H1_pooled"], M2: BASE + H["H2_by_type"], M3: BASE + H["H3_by_window"],
              M3T: BASE + H["H3_by_window"] + H["H1_pooled"]}
display(run_models(HISTORY_MODELS))
HISTORY_PAIRS = [(M1, "LambdaMART base"), (M2, M1), (M3, M2), (M3, M1), (M3T, M3), (M3T, M2), (FULL_NAME, M3T)]
paired_table(HISTORY_PAIRS)

By activity in the last 7 days: shoppers without recent activity have no clicks, so a type or window effect could look different for them.

In [ ]:
active = clean_ids[(searches.loc[clean_ids].days_since_last_action < 7).to_numpy()]
inactive = clean_ids.difference(active)
segment_pairs = [(M1, "LambdaMART base"), (M2, M1), (M3, M2), (M3T, M2)]
pd.concat({f"active in last 7 days ({len(active):,})": paired_table(segment_pairs, active),
           f"inactive 7+ days ({len(inactive):,})": paired_table(segment_pairs, inactive)})

In [ ]:
for name in HISTORY_MODELS:
    log_model("history representation", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
jd.read_results(paths).query("step == 'history representation'")

**Result.**
- **Pooled history alone takes almost the whole personal gain:** M1 reaches about **0.388**, about +0.041 over the base, against +0.042 for the deployable full model.
- **Splitting by action type or age adds nothing measurable on the dev subsample** (M2 minus M1 and M3 minus M2 about 0.0000); putting the pooled totals back (M3 + H1) changes nothing either. The same holds for active and inactive shoppers. By the decision rule, **the pooled representation is preferred**. (With all training data, section 11, type and window add a detectable but small amount, below the threshold.)
- The age split exists only for carts and orders (clicks are all recent, follows have no age block).

## 8. Common-window control for the action type

Action type and age are confounded: clicks are only recorded for 7 days, orders mostly come from months ago. A type effect could therefore be an age effect. Here both models only see the **last 7 days**, where all types are observed:

- **C-a:** base + pooled 0-7 d history, i.e. click + cart + order counts of the last 7 days per match level (`recent_0_7_*`, summed while loading; 4 inputs).
- **C-b:** base + the same evidence split by type (`click_0_7_*`, `cart_0_7_*`, `ord_0_7_*`; 12 inputs).

C-b minus C-a is the type effect without the age confound. Within 7 days orders are rare, so this mainly compares clicks with carts. Shoppers without activity in the last 7 days have only zeros here, so the difference is also shown for active shoppers alone.

In [ ]:
CA, CB = "C-a base + pooled 0-7 d", "C-b base + by type 0-7 d"
WINDOW_MODELS = {CA: BASE + [f"recent_0_7_{level}" for level in LEVELS],
              CB: BASE + [f"{block}_{level}" for block in ("click_0_7", "cart_0_7", "ord_0_7") for level in LEVELS]}
display(run_models(WINDOW_MODELS))
WINDOW_PAIRS = [(CA, "LambdaMART base"), (CB, "LambdaMART base"), (CB, CA)]
pd.concat({"validation_clean": paired_table(WINDOW_PAIRS),
           f"active in last 7 days ({len(active):,})": paired_table(WINDOW_PAIRS, active)})

In [ ]:
for name in WINDOW_MODELS:
    log_model("common window", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
jd.read_results(paths).query("step == 'common window'")

**What older history adds** (an additional comparison): M1 pools all ages, C-a only the last 7 days, with the same four match levels; the difference is the value of carts and orders older than 7 days (and follows of any age, which have no 7-day block) on top of recent evidence.

In [ ]:
pd.concat({"validation_clean": paired_table([(M1, CA)]),
           f"active in last 7 days ({len(active):,})": paired_table([(M1, CA)], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(M1, CA)], inactive)})

**Result.**
- Within the last 7 days, splitting by type adds about +0.002 (interval excluding 0), below the +0.003 threshold. **Action type is not an important dimension of history here**, once age is held fixed or all evidence is pooled.
- **Older history matters:** M1 (all ages) beats C-a (last 7 days) by about **+0.012**, about +0.019 for shoppers without recent activity. Older evidence carries real signal; it just does not need to be split by age.
- Together: use **all history, pooled over types and ages**, at four match levels.

## 9. Remaining feature groups added to pooled history

Starting from M1 (base + pooled history), each remaining group is **added on its own** and compared with M1 on the same searches:

| Group | Inputs | Question |
|---|---|---|
| H4 match recency | days since the last action on this product, shop, brand | does it matter *how long ago* the shopper touched it? |
| H5 metadata coverage | share of history with known metadata, per block | can the model discount matches where metadata is often missing? |
| C1 + C2 volume and context | history length and counts per block; time since the last action, actions in the last 7 days, query-less share, last action type | does the shopper's situation change how history is used? |
| time since the last action | `days_since_last_action` alone | the single most direct context signal |
| Q1 query history | overlap of the current query with past queries; matches in query-related history | should history count more when it relates to the query? |
| S1 history source | matches in search-linked vs query-less history | is history from searches better evidence than browsing? |

M1 + all six groups shows their joint value. Decision rule as in section 7. Search-level inputs (context, volume) act only through interactions with candidate features.

In [ ]:
DAYS = ["days_since_last_action"]
EXTRA_GROUPS = {"H4 match recency": H["H4_match_recency"], "H5 metadata coverage": H["H5_metadata_coverage"],
              "C1 + C2 volume and context": H["C1_history_volume"] + H["C2_session_context"],
              "time since last action": DAYS, "Q1 query history": H["Q1_query_history"],
              "S1 history source": H["S1_history_source"]}
GROUP_MODELS = {f"M1 + {group}": BASE + H["H1_pooled"] + columns for group, columns in EXTRA_GROUPS.items()}
M1_ALL = "M1 + all six groups"
GROUP_MODELS[M1_ALL] = BASE + H["H1_pooled"] + [c for group in EXTRA_GROUPS.values() for c in group if c not in DAYS] + DAYS
assert all(len(set(columns)) == len(columns) for columns in GROUP_MODELS.values())
display(run_models(GROUP_MODELS))
GROUP_PAIRS = [(name, M1) for name in GROUP_MODELS] + [(FULL_NAME, M1_ALL)]
paired_table(GROUP_PAIRS)

By activity in the last 7 days: context should matter most where recent evidence exists.

In [ ]:
pd.concat({f"active in last 7 days ({len(active):,})": paired_table([(n, M1) for n in GROUP_MODELS], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(n, M1) for n in GROUP_MODELS], inactive)})

In [ ]:
for name in GROUP_MODELS:
    log_model("feature groups", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
jd.read_results(paths).query("step == 'feature groups'")

**Result.**
- **No remaining group adds a practically relevant amount to pooled history** on the dev subsample: each group changes NDCG@10 by -0.0013 to +0.0002, all six together by about +0.0006; none meets the decision rule. With all training data, M1 + all six adds about +0.002 (section 15), still below the threshold.
- Time since the last action alone is slightly negative; a search-level input can only act through interactions, and this is most likely early-stopping noise rather than harm.
- **Session context and query overlap do not change how history should be used** once the history itself is in the model (also not for active shoppers). Effects below about 0.003 are not resolvable with this setup, so "no practically relevant gain" is the supported statement, not "no effect".
- Working LambdaMART: **base + pooled history (M1, 14 inputs)**.

## 10. Item-item collaborative features

All features so far match a candidate against the shopper's **own** history (same product, shop, brand, category). Collaborative filtering adds what **other shoppers** did: two products are similar if many shoppers have both in their history. A candidate the shopper never touched can then still score if it is similar to products they did touch.

**Graph**, from training histories only:
- events at least one day before their owner's search, distinct products, the 50 most recent per shopper;
- products with at least 5 shoppers; co-occurrence counts n_ij from one sparse matrix product (never a dense item-by-item matrix), diagonal set to zero (exact repeats stay in H1), pairs with n_ij >= 2;
- similarity n_ij / sqrt(n_i n_j) x n_ij / (n_ij + 10): cosine, shrunk towards 0 for pairs seen by few shoppers; the 50 most similar neighbours per product.

**3-fold cross-fitting.** A training search's own history is in the graph, which would make its similarities look more informative than a validation search's. Training searches therefore get a fold (seeded per search ID), and their features use the graph built from the other two folds; validation and test use the graph from all training searches.

**Features per candidate**, over the shopper's distinct history products (all ages): sum, maximum and mean similarity (mean over the shopper's products in the graph) and the number of history products among the candidate's neighbours; plus the number of the shopper's products in the graph (search level). Candidates outside the graph are missing (unknown), not 0. Pooled only, since type and age splits did not help (section 7). Features are written to `processed/cf/`; changing `CF_SPEC` rebuilds them.

In [ ]:
CF_SPEC = {"version": "cf1", "min_age_days": 1, "cap": 50, "min_support": 5, "min_pair": 2, "shrink": 10,
           "top_k": 50, "folds": 3}
CF_COLUMNS = ["cf_sum", "cf_max", "cf_mean", "cf_count", "cf_history_in_graph"]
CF_SPLITS = {"train_dev": dev_ids.to_numpy(), "validation": validation_ids.to_numpy(),
             "train_rest": assignment.index[(assignment.split == "train") & ~assignment.dev_subsample].to_numpy(),
             "test": assignment.index[assignment.split == "test"].to_numpy()}  # features only, for notebook 07
CF_TRAINING = {"train_dev", "train_rest"}  # cross-fitted; the other splits use the graph from all training searches
CF_DIR = PROCESSED / "cf"


def cf_name(version=None):
    """CF files of the v3 features are `cf1_*`; the safety-gap version (section 12) gets its own prefix."""
    version = version or FEATURES
    return CF_SPEC["version"] if version == FEATURES else f"{CF_SPEC['version']}_{version}"


def cf_file(split, version=None):
    return CF_DIR / f"{cf_name(version)}_{split}.parquet"


def read_histories(splits, target_min_age_days=0.0):
    """Graph histories of all training searches (lagged, distinct, most recent first) and target histories.

    Target histories keep the actions at least `target_min_age_days` before their search (0: all; section 12: 10 min).
    """
    train_set = set(assignment.index[assignment.split == "train"])
    targets = set(np.concatenate([CF_SPLITS[s] for s in splits]).tolist())
    graph, target = {}, {}
    for sid, fields in iter_records(BEHAVIOUR):
        if sid not in train_set and sid not in targets:
            continue
        past = np.array(fields[4].split("_"), dtype=np.int64)
        gaps = np.array(fields[6].split("_"), dtype=np.int64)
        age = np.cumsum(gaps[::-1])[::-1][1:] / SECONDS_PER_DAY
        if sid in train_set:
            recent_first = past[::-1][age[::-1] >= CF_SPEC["min_age_days"]]
            _, first = np.unique(recent_first, return_index=True)
            graph[sid] = recent_first[np.sort(first)]
        if sid in targets:
            target[sid] = np.unique(past[age >= target_min_age_days])
    return graph, target


def lookup(vocab, items):
    """Positions of `items` in the sorted `vocab`, and which items are in it."""
    pos = np.searchsorted(vocab, items)
    found = (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == items)
    return pos, found


def build_graph(item_lists, block=40_000):
    """Sorted product vocabulary and top-k similarity matrix (CSR, one row of neighbours per product).

    The co-occurrence counts are computed in blocks of `block` products (rows), each thresholded and cut to its top-k
    neighbours right away, so the full count matrix never sits in memory.
    """
    items, counts = np.unique(np.concatenate(item_lists), return_counts=True)
    vocab = items[counts >= CF_SPEC["min_support"]]
    rows, cols = [], []
    for u, lst in enumerate(item_lists):
        pos, found = lookup(vocab, lst)
        pos = pos[found][:CF_SPEC["cap"]]  # lists are most recent first
        rows.append(np.full(len(pos), u, np.int32))
        cols.append(pos.astype(np.int32))
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    X = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(item_lists), len(vocab)))
    del rows, cols
    n_i = np.asarray(X.sum(axis=0)).ravel()
    Xt = X.T.tocsr()  # products x shoppers
    k, n = CF_SPEC["top_k"], len(vocab)
    indptr, indices, data, pairs = np.zeros(n + 1, np.int64), [], [], 0
    for lo in range(0, n, block):
        hi = min(lo + block, n)
        co = (Xt[lo:hi] @ X).tocsr()  # co-occurrence counts of products lo..hi with all products
        row_of = np.repeat(np.arange(lo, hi, dtype=np.int32), np.diff(co.indptr))
        co.data[(co.indices == row_of) | (co.data < CF_SPEC["min_pair"])] = 0  # diagonal, rare pairs
        co.eliminate_zeros()
        pairs += co.nnz
        row_of = np.repeat(np.arange(lo, hi, dtype=np.int32), np.diff(co.indptr))
        n_ij = co.data
        sims = (n_ij / np.sqrt(n_i[row_of] * n_i[co.indices]) * n_ij / (n_ij + CF_SPEC["shrink"])).astype(np.float32)
        for r in range(hi - lo):
            a, b = co.indptr[r], co.indptr[r + 1]
            d, c = sims[a:b], co.indices[a:b]
            if b - a > k:
                keep = np.argpartition(-d, k)[:k]
                d, c = d[keep], c[keep]
            indices.append(c)
            data.append(d)
            indptr[lo + r + 1] = indptr[lo + r] + len(c)
        del co, row_of, n_ij, sims
    del X, Xt
    S = sparse.csr_matrix((np.concatenate(data), np.concatenate(indices), indptr), shape=(n, n))
    return vocab, S, {"products": n, "pairs_before_top_k": int(pairs), "neighbours": int(S.nnz)}


def score(out, sid, wids, rows, vocab, S, history, mask):
    """Fill the CF features of one search (rows of `out`) from graph (vocab, S)."""
    hist_pos, hist_found = lookup(vocab, history)
    hist_pos = hist_pos[hist_found]
    out[rows, 4] = len(hist_pos)
    mask[hist_pos] = True
    pos, found = lookup(vocab, wids)
    for r, i in zip(rows[found], pos[found]):
        lo, hi = S.indptr[i], S.indptr[i + 1]
        sims = S.data[lo:hi][mask[S.indices[lo:hi]]]
        out[r, :4] = (sims.sum(), sims.max() if len(sims) else 0.0,
                      sims.sum() / len(hist_pos) if len(hist_pos) else np.nan, len(sims))
    mask[hist_pos] = False

In [ ]:
def build_cf_features(version, splits, target_min_age_days=0.0):
    """Build (or keep) the CF feature files of `splits` for a feature version; returns the graph statistics."""
    spec = CF_SPEC if target_min_age_days == 0 else {**CF_SPEC, "target_min_age_days": target_min_age_days}
    spec_file, stats_file = CF_DIR / f"{cf_name(version)}_spec.json", CF_DIR / f"{cf_name(version)}_graphs.json"
    spec_ok = not RETRAIN and spec_file.is_file() and json.loads(spec_file.read_text()) == spec
    missing = [s for s in splits if not (spec_ok and cf_file(s, version).is_file())]  # only these are built
    if missing:
        start = time.time()
        graph_histories, target_histories = read_histories(missing, target_min_age_days)
        cf_fold = {sid: int(np.random.default_rng([SEED, 22, sid]).integers(CF_SPEC["folds"])) for sid in graph_histories}
        print(f"histories read in {time.time() - start:.0f} s: {len(graph_histories):,} graph, {len(target_histories):,} target")
        tables = {split: (ids, np.full((len(ids), len(CF_COLUMNS)), np.nan, dtype=np.float32))
                  for split in missing
                  for ids in [pq.read_table(feature_file(split, version), columns=["session_id", "wid"]).to_pandas()]}
        graph_stats = json.loads(stats_file.read_text()) if spec_ok and stats_file.is_file() else {}
        # one graph per fold for training searches (graph from the other folds), one from all training searches
        training, other = [s for s in missing if s in CF_TRAINING], [s for s in missing if s not in CF_TRAINING]
        jobs = [(f"fold {f}", [s for s in graph_histories if cf_fold[s] != f], training, f)
                for f in range(CF_SPEC["folds"])] if training else []
        if other:
            jobs.append(("all training", list(graph_histories), other, None))
        for name, graph_ids, job_splits, fold in jobs:
            start = time.time()
            vocab, S, stats = build_graph([graph_histories[s] for s in graph_ids])
            mask = np.zeros(len(vocab), dtype=bool)
            for split in job_splits:
                ids, out = tables[split]
                sid, wid = ids.session_id.to_numpy(), ids.wid.to_numpy()
                bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
                for lo, hi in zip(bounds[:-1], bounds[1:]):
                    if fold is None or cf_fold[sid[lo]] == fold:
                        score(out, sid[lo], wid[lo:hi], np.arange(lo, hi), vocab, S, target_histories[sid[lo]], mask)
            graph_stats[name] = {**stats, "graph_searches": len(graph_ids), "seconds": round(time.time() - start),
                                 "peak_memory_gb": round(memory_gb()[1], 2)}
            del vocab, S, mask
            gc.collect()
        CF_DIR.mkdir(parents=True, exist_ok=True)
        for split, (ids, out) in tables.items():
            assert not np.isnan(out[:, 4]).any()  # every search was scored by exactly one graph
            frame = ids.copy()
            for j, column in enumerate(CF_COLUMNS):
                frame[column] = out[:, j]
            frame.to_parquet(cf_file(split, version), index=False)
        spec_file.write_text(json.dumps(spec))
        stats_file.write_text(json.dumps(graph_stats))
        del graph_histories, target_histories, tables
        gc.collect()
    return json.loads(stats_file.read_text())


graph_stats = build_cf_features(FEATURES, list(CF_SPLITS))
for split in CF_SPLITS:  # the CF rows must line up with the v3 rows
    cf_ids = pq.read_table(cf_file(split), columns=["session_id", "wid"]).to_pandas()
    v3_ids = pq.read_table(feature_file(split), columns=["session_id", "wid"]).to_pandas()
    assert cf_ids.equals(v3_ids)
EXTRA.update({column: cf_file for column in CF_COLUMNS})
pd.DataFrame(graph_stats).T

Coverage: share of candidates in the graph (CF features observed) and with at least one similar product in the shopper's history. For training rows also among engaged candidates (training labels only).

In [ ]:
rows = {}
for split in ("train_dev", "validation"):
    cf = pq.read_table(cf_file(split), columns=["session_id", "cf_sum", "cf_count"]).to_pandas()
    keep = cf.session_id.isin(train_pool if split == "train_dev" else validation_ids).to_numpy()
    cf = cf[keep]
    rows[split] = {"candidates": len(cf), "in graph": cf.cf_sum.notna().mean(), "similar product in history": (cf.cf_count > 0).mean()}
    if split == "train_dev":
        labels = pq.read_table(FEATURE_FILES[0], columns=["label"]).column(0).to_numpy()[keep]
        engaged = labels >= 1
        rows["train_dev, engaged"] = {"candidates": int(engaged.sum()), "in graph": cf.cf_sum.notna()[engaged].mean(),
                                      "similar product in history": (cf.cf_count[engaged] > 0).mean()}
pd.DataFrame(rows).T

In [ ]:
M1_CF = "M1 + CF"
CF_MODELS = {M1_CF: BASE + H["H1_pooled"] + CF_COLUMNS}
display(run_models(CF_MODELS))
pd.concat({"validation_clean": paired_table([(M1_CF, M1), (M1_CF, "LambdaMART base")]),
           f"active in last 7 days ({len(active):,})": paired_table([(M1_CF, M1)], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(M1_CF, M1)], inactive)})

In [ ]:
cf_model, _, _ = train_ranker(CF_MODELS[M1_CF], label=M1_CF)  # loaded from the cache
importance = pd.Series(cf_model.feature_importance("gain"), index=CF_MODELS[M1_CF])
display((importance / importance.sum()).sort_values(ascending=False).head(10).rename("gain share").to_frame())
log_model("collaborative features", M1_CF, "validation_clean", per_search[M1_CF].loc[clean_ids], model_summaries[M1_CF])
jd.read_results(paths).query("step == 'collaborative features'")

**Result.**
- **CF adds about +0.002** to pooled history on the dev subsample: an interval excluding 0, but below the +0.003 threshold.
- **Why so little: the links are sparse.** Only 15-19% of candidates are in the graph and only **2-3% have a similar product in the shopper's history** (9% of engaged training candidates). After requiring two shared shoppers per pair, most products have few or no neighbours, so for most candidates CF has nothing to say beyond popularity.

## 11. Final models on all training searches

Sections 3-10 were screening on the 40,000-search dev subsample. The finalists are now retrained on **all clean training searches** (the dev subsample plus `train_rest`), with the same inner early-stopping set and configuration. **These are the reported numbers.** LightGBM without bagging is deterministic, so one run per model.

Finalists: the deployable base, M1 (base + pooled history), M3 (history by type and window), M1 + CF and the deployable full model (all 94 deployable inputs). Inputs above 1.5 GB (M3 and the full model) are loaded into a memory-mapped file, so the 11.3M training rows do not need to fit in free memory.

In [ ]:
train_info = searches.loc[assignment.index[assignment.split == "train"]]
full_pool = train_info.index[(~train_info.final_gap_clamped & (train_info.n_engaged > 0)
                              & (train_info.n_candidates > 1)).to_numpy()]
full_fit_ids = np.setdiff1d(full_pool.to_numpy(), inner_ids)
ALL_TRAINING = ("train_dev", "train_rest")
FINALISTS = {"base": (BASE, "LambdaMART base"), "M1": (BASE + H["H1_pooled"], M1),
             "M3": (BASE + H["H3_by_window"], M3), "M1 + CF": (BASE + H["H1_pooled"] + CF_COLUMNS, M1_CF),
             "deployable full": (DEPLOYABLE_FULL, FULL_NAME)}
print(f"training pool {len(full_pool):,} searches; fit {len(full_fit_ids):,}; inner early-stopping set {len(inner_ids):,}")
rows = {}
for short, (columns, dev_name) in FINALISTS.items():
    name = f"{short}, all training"
    _, summary, scores = train_ranker(columns, label=name, fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    model_summaries[name] = summary
    d = paired_difference(per_search[name].loc[clean_ids], per_search[dev_name].loc[clean_ids])
    rows[short] = {"inputs": len(columns), "best_iteration": summary["best_iteration"],
                   "train_seconds": summary["train_seconds"], "peak_memory_gb": summary["peak_memory_gb"],
                   "cached": summary["cached"], "dev_trained": per_search[dev_name].ndcg.loc[clean_ids].mean(),
                   "all_training": per_search[name].ndcg.loc[clean_ids].mean(),
                   "all_training_all_searches": per_search[name].ndcg.mean(),
                   "gain_from_data": d[0], "low": d[1], "high": d[2]}
pd.DataFrame(rows).T

The comparisons that carry the conclusions, on the full-data models (paired, `validation_clean`):

In [ ]:
F = {short: f"{short}, all training" for short in FINALISTS}
RULE_A = "rule (a): own-product count, then popularity"
FINAL_PAIRS = [(F["base"], "lagged history popularity"), (F["M1"], F["base"]), (F["M1"], RULE_A),
             (F["M1"], "lagged history popularity"), (F["M3"], F["M1"]), (F["M1 + CF"], F["M1"]),
             (F["deployable full"], F["M1"])]
display(paired_table(FINAL_PAIRS))
pd.concat({f"active in last 7 days ({len(active):,})": paired_table([(F["M1"], F["base"])], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(F["M1"], F["base"])], inactive)})

In [ ]:
for short in FINALISTS:
    for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
        log_model("final models", F[short], split, per_search[F[short]].loc[ids], model_summaries[F[short]])
jd.read_results(paths).query("step == 'final models'")

**Result** (`validation_clean`, 121,989 fit searches).
- **More data helps only a little:** about +0.001 to +0.003 per model over the dev-trained versions.
- **Pooled history carries the personal gain:** M1 about **0.389**, about **+0.039** over the base (0.350; active shoppers about +0.042, inactive about +0.018), and above the untrained rule. Splitting history by type and window (M3) adds about +0.0013 and the full model about +0.0019: detectable, below the threshold.
- **M1 + CF reaches about 0.392**, about +0.003 over M1, exactly at the decision threshold in this deterministic run. Across three seeds the CF gain stays below it (section 15), so CF is **no robust gain**: M1 + CF is the final LambdaMART by the decision rule, with M1 as an equal headline.
- Deployable base vs lagged popularity: about +0.0125.

## 12. Robustness: history from at least 10 minutes before the search

In 5% of searches the history contained actions from after the search (the clamped final gap); those searches are excluded, but similar effects could hide in the rest. Here every history action younger than **10 minutes** before the search is dropped, both in the history and context features (the safety-gap version of notebook 03, `v3_minage0.00694444d`, built for `train_dev` and `validation`) and on the shopper's side of the CF features (the graph already uses only actions at least one day old). Popularity and relevance do not change. M1 and M1 + CF are retrained on the dev subsample with these features and compared with their normal versions.

A drop is **not proof of leakage**: actions minutes before a search are legitimate evidence mid-session. The question is whether the personal gain stays clearly positive.

In [ ]:
GAP_DAYS = 10 / (24 * 60)
GAP = f"v3_minage{GAP_DAYS:g}d"  # the safety-gap version written by notebook 03
for split in ("train_dev", "validation"):  # same searches, candidates and order as v3; base inputs unchanged
    v3 = pq.read_table(feature_file(split), columns=["session_id", "wid"] + BASE).to_pandas()
    gap = pq.read_table(feature_file(split, GAP), columns=["session_id", "wid"] + BASE).to_pandas()
    assert v3.equals(gap), split
gap_meta = json.loads((PROCESSED / f"features_{GAP}_meta.json").read_text())
assert gap_meta["groups"] == meta["groups"] and abs(gap_meta["min_age_days"] - GAP_DAYS) < 1e-12
gap_stats = build_cf_features(GAP, ["train_dev", "validation"], GAP_DAYS)
for split in ("train_dev", "validation"):
    assert pq.read_table(cf_file(split, GAP), columns=["session_id", "wid"]).to_pandas().equals(
        pq.read_table(feature_file(split), columns=["session_id", "wid"]).to_pandas())
# how much history the gap removes (validation, clean searches)
rows = {}
for name, version in [("v3", None), ("10-minute gap", GAP)]:
    t = pq.read_table(feature_file("validation", version), columns=["session_id", "hist_len", "pooled_product"]).to_pandas()
    t = t[t.session_id.isin(clean_ids)]
    per = t.groupby("session_id").agg(hist_len=("hist_len", "first"), product=("pooled_product", "sum"))
    rows[name] = {"history actions per search": per.hist_len.mean(), "exact-product matches per search": per["product"].mean(),
                  "searches with an exact-product match": (per["product"] > 0).mean()}
pd.DataFrame(rows).T

In [ ]:
M1_GAP, M1_CF_GAP = "M1, 10-minute gap", "M1 + CF, 10-minute gap"
GAP_MODELS = {M1_GAP: HISTORY_MODELS[M1], M1_CF_GAP: CF_MODELS[M1_CF]}
for name, columns in GAP_MODELS.items():
    _, summary, scores = train_ranker(columns, label=name, version=GAP)
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    model_summaries[name] = summary
clamped_ids = all_ids.difference(clean_ids)
GAP_PAIRS = [(M1_GAP, M1), (M1_CF_GAP, M1_CF), (M1_GAP, "LambdaMART base"), (M1_CF_GAP, "LambdaMART base"),
             (M1_CF_GAP, M1_GAP)]
pd.concat({"validation_clean": paired_table(GAP_PAIRS),
           f"active in last 7 days ({len(active):,})": paired_table(GAP_PAIRS[:2], active),
           f"all searches ({len(all_ids):,})": paired_table(GAP_PAIRS[:2], all_ids),
           f"clamped only ({len(clamped_ids):,})": paired_table(GAP_PAIRS[:2], clamped_ids)})

In [ ]:
for name in GAP_MODELS:
    log_model("robustness: 10-minute gap", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name], version=GAP)
jd.read_results(paths).query("step == 'robustness: 10-minute gap'")

**Result.**
- The gap removes few actions but much exact-product evidence: about 2 of 154 history actions per validation search, but about 23% of the exact-product matches. Shoppers often look at a product minutes before searching for it.
- **Without them M1 + CF loses about 0.0125** on the dev subsample (M1 about 0.013). **The personal gain stays clearly positive:** about +0.030 above the base with the gap (normally about +0.043). On the clamped searches the drop is much larger, as expected for post-search actions.
- Only searches with a final gap under 10 minutes can change (a longer final gap leaves no action younger than 10 minutes); the split by final gap is in section 15. The loss is a sensitivity of the retrained model: it does not tell session evidence from residual post-search actions, and only M1 and M1 + CF were retrained with the gap.

## 13. CF graph-size check

Training rows get their CF features from a **fold graph** (built on the two other folds, about two thirds of the training searches), validation rows from the graph of **all** training searches (section 10). The larger graph covers more products, so validation CF features are distributed slightly differently from the ones the models were trained on.

Check without retraining: recompute the validation CF features with each of the three fold graphs (the same graphs as for training rows) and with their average, re-score the trained M1 + CF models (dev subsample and all training searches) and compare with their full-graph scores. Decision rule: if the result moves by 0.002 or more, validation and test CF features would come from the fold graphs instead.

In [ ]:
import warnings

variant_file = CF_DIR / f"{CF_SPEC['version']}_validation_fold_graphs.npz"
val_ids = pq.read_table(feature_file("validation"), columns=["session_id", "wid"]).to_pandas()
if RETRAIN or not variant_file.is_file():
    graph_histories, target_histories = read_histories(["validation"])
    cf_fold = {sid: int(np.random.default_rng([SEED, 22, sid]).integers(CF_SPEC["folds"])) for sid in graph_histories}
    sid, wid = val_ids.session_id.to_numpy(), val_ids.wid.to_numpy()
    bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
    arrays = {}
    for f in range(CF_SPEC["folds"]):
        vocab, S, _ = build_graph([graph_histories[s] for s in graph_histories if cf_fold[s] != f])
        out = np.full((len(val_ids), len(CF_COLUMNS)), np.nan, dtype=np.float32)
        mask = np.zeros(len(vocab), dtype=bool)
        for lo, hi in zip(bounds[:-1], bounds[1:]):
            score(out, sid[lo], wid[lo:hi], np.arange(lo, hi), vocab, S, target_histories[sid[lo]], mask)
        arrays[f"fold_{f}"] = out
        del vocab, S, mask
    np.savez(variant_file, **arrays)
    del graph_histories, target_histories
    gc.collect()
loaded = np.load(variant_file)
cf_variants = {f"fold graph {f}": loaded[f"fold_{f}"] for f in range(CF_SPEC["folds"])}
with warnings.catch_warnings():  # products in no fold graph stay missing
    warnings.simplefilter("ignore", category=RuntimeWarning)
    cf_variants["average of the fold graphs"] = np.nanmean(np.stack(list(cf_variants.values())), axis=0)
full_cf = pq.read_table(cf_file("validation"), columns=CF_COLUMNS).to_pandas().to_numpy(dtype=np.float32)
pd.DataFrame({name: {"in graph": np.isfinite(a[:, 0]).mean(), "similar product in history": (a[:, 3] > 0).mean(),
                     "mean cf_sum (in graph)": np.nanmean(a[:, 0])}
              for name, a in {"full graph (used so far)": full_cf, **cf_variants}.items()}).T

In [ ]:
GRAPH_CHECK_MODELS = {"dev subsample": (M1_CF, M1, {}), "all training": (F["M1 + CF"], F["M1"],
                                                                    {"fit_sessions": full_fit_ids, "train_splits": ALL_TRAINING})}
columns = CF_MODELS[M1_CF]
cf_positions = [columns.index(c) for c in CF_COLUMNS]
_, X_val, _ = load_features("validation", columns)
rows = {}
for model, (name, m1_name, kwargs) in GRAPH_CHECK_MODELS.items():
    booster, _, cached_scores = train_ranker(columns, label=name, **kwargs)  # loaded from the cache
    assert np.allclose(booster.predict(X_val), cached_scores.score.to_numpy(), atol=1e-9)  # full graph reproduces
    for variant, values in cf_variants.items():
        X = X_val.copy(order="F")
        X[:, cf_positions] = values
        key = f"{name}, CF from {variant}"
        per_search[key] = evaluate(validation_frame.assign(score=booster.predict(X)), "score",
                                   expected_sizes=validation_sizes)
        vs_full = paired_difference(per_search[key].loc[clean_ids], per_search[name].loc[clean_ids])
        vs_m1 = paired_difference(per_search[key].loc[clean_ids], per_search[m1_name].loc[clean_ids])
        rows[(model, variant)] = {"ndcg_clean": per_search[key].ndcg.loc[clean_ids].mean(),
                                  "vs_full_graph": vs_full[0], "low": vs_full[1], "high": vs_full[2],
                                  "vs_M1": vs_m1[0], "vs_M1_low": vs_m1[1], "vs_M1_high": vs_m1[2]}
    rows[(model, "full graph (used so far)")] = {"ndcg_clean": per_search[name].ndcg.loc[clean_ids].mean(),
                                                 **dict(zip(["vs_M1", "vs_M1_low", "vs_M1_high"], paired_difference(
                                                     per_search[name].loc[clean_ids], per_search[m1_name].loc[clean_ids])))}
del X_val
pd.DataFrame(rows).T

**Result.**
- Fold graphs cover fewer validation candidates (about 15.5% vs 18.7% in the graph), as on training rows. Re-scoring the trained M1 + CF models with fold-graph CF features changes `validation_clean` by less than 0.001; the average of the three fold graphs gives the same result as the full graph. Below the 0.002 rule: **validation and test keep the full-graph CF features**, also for the neural models.
- Side result: with fold-graph features, CF's gain over M1 on all training searches is slightly smaller (about +0.0027 instead of +0.0031): CF is a gain right at the threshold, not a robust one.

## 14. Export for the neural models

Notebook 05b compares its models with LambdaMART on the same searches. It reads the validation scores of the full-data base, M1 and M1 + CF from `processed/neural/lambdamart_validation_scores.parquet`, one row per validation candidate in the order of the v3 file.

In [ ]:
NEURAL_DIR = PROCESSED / "neural"
NEURAL_DIR.mkdir(parents=True, exist_ok=True)
reference = pq.read_table(feature_file("validation"), columns=["session_id", "wid"]).to_pandas()
for short, key in [("base", "base"), ("m1", "M1"), ("m1_cf", "M1 + CF")]:
    _, _, scores = train_ranker(FINALISTS[key][0], label=F[key], fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
    assert (scores.session_id.to_numpy() == reference.session_id.to_numpy()).all()
    reference[short] = scores.score.to_numpy()
reference.to_parquet(NEURAL_DIR / "lambdamart_validation_scores.parquet", index=False)
reference.describe().loc[["count", "mean", "std"]]

## 15. Sensitivity checks

Four checks on the full-data models.

**(a) A fairer untrained rule.** Rule (a) of section 6 ranks own products first, then by lagged popularity; the trained base also uses relevance. Here own products come first and the **trained base's score** orders the rest, so the comparison with M1 isolates what learning adds to "known products first".

In [ ]:
own_count = pq.read_table(feature_file("validation"), columns=["pooled_product"]).column(0).to_numpy()
_, _, base_all = train_ranker(FINALISTS["base"][0], label=F["base"], fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
b = base_all.score.to_numpy()
RULE_BASE = "rule: own products first, then the trained base"
per_search[RULE_BASE] = evaluate(validation_frame.assign(score=np.nan_to_num(own_count) + 0.999 * (b - b.min()) / (b.max() - b.min())),
                                 "score", expected_sizes=validation_sizes)
display(paired_table([(RULE_BASE, F["base"]), (F["M1"], RULE_BASE), (F["M1 + CF"], RULE_BASE), (RULE_BASE, RULE_A)]))
share = ((per_search[RULE_BASE].ndcg.loc[clean_ids].mean() - per_search[F["base"]].ndcg.loc[clean_ids].mean())
         / (per_search[F["M1"]].ndcg.loc[clean_ids].mean() - per_search[F["base"]].ndcg.loc[clean_ids].mean()))
print(f"share of M1's gain over the base reached by the rule: {share:.1%}")

**(b) Final-gap sensitivity.** Searches whose final gap is 2-5 s behave like the clamped ones, and 6-30 s is still elevated. The final models are re-evaluated without them (no retraining), and the safety-gap loss of section 12 is split by final gap.

In [ ]:
gap = searches.final_gap_s.reindex(clean_ids)
populations = {"validation_clean": clean_ids, "without final gap <= 5 s": clean_ids[(gap > 5).to_numpy()],
               "without final gap <= 30 s": clean_ids[(gap > 30).to_numpy()]}
rows = {}
for population, ids in populations.items():
    row = {"searches": len(ids)}
    for short in ("base", "M1", "M1 + CF"):
        row[short] = per_search[F[short]].ndcg.loc[ids].mean()
    d = paired_difference(per_search[F["M1"]].loc[ids], per_search[F["base"]].loc[ids])
    row.update(M1_minus_base=d[0], low=d[1], high=d[2])
    rows[population] = row
display(pd.DataFrame(rows).T)
buckets = pd.cut(gap, [1, 5, 30, 60, 600, 3600, np.inf], labels=["2-5 s", "6-30 s", "31-60 s", "1-10 min", "10-60 min", "over 1 h"])
rows = []
for value in buckets.cat.categories:
    ids = buckets.index[buckets == value]
    d = paired_difference(per_search[M1_CF_GAP].loc[ids], per_search[M1_CF].loc[ids])
    rows.append({"final gap": value, "searches": len(ids), "safety-gap loss (gap minus normal)": d[0], "low": d[1], "high": d[2],
                 "share of the total loss": d[0] * len(ids) / ((per_search[M1_CF_GAP].ndcg.loc[clean_ids]
                                                               - per_search[M1_CF].ndcg.loc[clean_ids]).sum())})
pd.DataFrame(rows).set_index("final gap")

**(c) Sensitivity to feature subsampling and seed.** LightGBM without subsampling is deterministic, so the bootstrap intervals only cover validation sampling. Here M1 and M1 + CF are retrained on all training searches with `feature_fraction` 0.8 and three seeds; CF minus M1 per seed shows how much of the 0.003 is robust. This changes the configuration and the seed together.

In [ ]:
seed_rows = []
for seed in (1, 2, 3):
    params = {**PARAMS, "feature_fraction": 0.8, "seed": seed}
    for short in ("M1", "M1 + CF"):
        name = f"{short}, all training, feature_fraction 0.8, seed {seed}"
        _, summary, scores = train_ranker(FINALISTS[short][0], params=params, label=name, fit_sessions=full_fit_ids,
                                          train_splits=ALL_TRAINING)
        per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
        model_summaries[name] = summary
    m1, cf = (f"{s}, all training, feature_fraction 0.8, seed {seed}" for s in ("M1", "M1 + CF"))
    d = paired_difference(per_search[cf].loc[clean_ids], per_search[m1].loc[clean_ids])
    seed_rows.append({"seed": seed, "M1": per_search[m1].ndcg.loc[clean_ids].mean(),
                      "M1 + CF": per_search[cf].ndcg.loc[clean_ids].mean(),
                      "best iterations": f"{model_summaries[m1]['best_iteration']} / {model_summaries[cf]['best_iteration']}",
                      "CF minus M1": d[0], "low": d[1], "high": d[2]})
seeds = pd.DataFrame(seed_rows).set_index("seed")
display(seeds)
print(f"CF minus M1 over seeds: mean {seeds['CF minus M1'].mean():+.4f}, range {seeds['CF minus M1'].min():+.4f} "
      f"to {seeds['CF minus M1'].max():+.4f}; M1 across seeds sd {seeds['M1'].std():.4f}")

**(d) M1 + all six groups of section 9 on all training searches.**

In [ ]:
M1_ALL_FULL = "M1 + all six groups, all training"
_, summary, scores = train_ranker(GROUP_MODELS[M1_ALL], label=M1_ALL_FULL, fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
per_search[M1_ALL_FULL] = evaluate(scores, "score", expected_sizes=validation_sizes)
model_summaries[M1_ALL_FULL] = summary
display(paired_table([(M1_ALL_FULL, F["M1"]), (F["deployable full"], M1_ALL_FULL)]))
jd.log_result(paths, "05", "sensitivity checks", RULE_BASE, "validation_clean", per_search[RULE_BASE].loc[clean_ids],
              notes="untrained: pooled_product count first, then the all-data base score (features v3)")
log_model("sensitivity checks", M1_ALL_FULL, "validation_clean", per_search[M1_ALL_FULL].loc[clean_ids],
          model_summaries[M1_ALL_FULL])
for seed in (1, 2, 3):
    for short in ("M1", "M1 + CF"):
        name = f"{short}, all training, feature_fraction 0.8, seed {seed}"
        log_model("sensitivity checks", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name],
                  seed=seed)
jd.read_results(paths).query("step == 'sensitivity checks'")[["model", "searches", "ndcg_at_10", "commit"]]

**Result.**
- **(a)** Own products first, then the trained base: about **0.384**, about 88% of M1's gain over the base. Learning adds about **+0.005** (M1) and +0.008 (M1 + CF) beyond "known products first".
- **(b)** Without final gaps up to 5 s nothing changes; **without final gaps up to 30 s (9% of searches) the personal gain is about +0.033 instead of +0.039**. Whether these short gaps carry session evidence or post-search actions cannot be decided; both numbers are reported. The safety-gap loss comes from short final gaps, about 41% of it from 6-30 s.
- **(c)** CF minus M1 across three seeds stays below +0.003 (mean about +0.001): **CF is no robust gain**. Feature subsampling moves M1 itself by about 0.001-0.002: differences of that size can come from training choices alone.
- **(d)** M1 + all six groups with all training data: about +0.002 over M1, below the threshold; the full 94-input model adds nothing beyond it.

## 16. Test scores (no labels)

The final models score the 20,000 test searches here, once, with the models trained above (loaded from the cache). Only features are read (the label column is never loaded); the scores are saved to `processed/test/lambdamart_test_scores.parquet`, and notebook 07 evaluates them after checking the validation numbers. CF features for test come from the graph of all training searches, like validation (section 10).

In [ ]:
TEST_DIR = PROCESSED / "test"
TEST_DIR.mkdir(parents=True, exist_ok=True)
TEST_MODELS = {"base": FINALISTS["base"][0], "M1": FINALISTS["M1"][0], "M3": FINALISTS["M3"][0],
               "M1 + CF": FINALISTS["M1 + CF"][0], "deployable full": FINALISTS["deployable full"][0],
               "M1 + all six": GROUP_MODELS[M1_ALL]}
test_scores = pq.read_table(feature_file("test"), columns=["session_id", "wid", "pooled_product"]).to_pandas()
for short, columns in TEST_MODELS.items():
    booster, summary, _ = train_ranker(columns, label=f"{short}, all training", fit_sessions=full_fit_ids,
                                       train_splits=ALL_TRAINING)  # the model trained above, from the cache
    assert summary["cached"], short
    sid, X, _ = load_features("test", columns, with_labels=False)
    assert (sid == test_scores.session_id.to_numpy()).all()
    test_scores[short] = booster.predict(X)  # saved boosters keep their best iteration only
    del X
    gc.collect()
remove_memmaps()
test_scores.to_parquet(TEST_DIR / "lambdamart_test_scores.parquet", index=False)
print(f"test: {test_scores.session_id.nunique():,} searches, {len(test_scores):,} candidates, models {list(TEST_MODELS)}")
test_scores.describe().loc[["count", "mean", "std"]]